In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pii_utils import PiiCrypto


storage_account_name = "stdevnortheuropebfn0"
base_path = f"abfss://data@{storage_account_name}.dfs.core.windows.net"

yellow_source_path = f"{base_path}/landing/yellow"
green_source_path = f"{base_path}/landing/green"
zones_source_path = f"{base_path}/reference/taxi_zone_lookup.csv"

pii_key = dbutils.secrets.get(scope="pii_scope", key="pii_aes_key")
crypto = PiiCrypto(encryption_key=pii_key)

pii_columns = [
    "customer_first_name",
    "customer_last_name",
    "customer_email"
]


def prepare_bronze(df, pickup_column):
    df = (
        df
        .withColumn("year", F.year(pickup_column))
        .withColumn("month", F.month(pickup_column))
        .withColumn("day", F.dayofmonth(pickup_column))
    )

    return crypto.encrypt_columns(df, pii_columns)

In [0]:
@dp.table(
    name="bronze.yellow_trips_raw",
    comment="Raw encrypted Yellow Taxi records",
    partition_cols=["year", "month", "day"]
)
def yellow_trips_raw():
    yellow_raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .load(yellow_source_path)
    )

    return prepare_bronze(
        yellow_raw_df,
        "tpep_pickup_datetime"
    )


@dp.table(
    name="bronze.green_trips_raw",
    comment="Raw encrypted Green Taxi records",
    partition_cols=["year", "month", "day"]
)
def green_trips_raw():
    green_raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .load(green_source_path)
    )

    return prepare_bronze(
        green_raw_df,
        "lpep_pickup_datetime"
    )


@dp.materialized_view(
    name="bronze.taxi_zones_raw",
    comment="Static NYC Taxi zone reference data"
)
def taxi_zones_raw():
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(zones_source_path)
    )